### Generative Adversarial Network(GAN):
- GAN help machines to create new, realistic data by learning from existing examples.
- They have transformed how computers generate images, videos, music and more.
### Architecture of GAN:
- GAN consist of two main models that work together to create realistic synthetic data which are as follows -
#### 1. Generator Model:
- The generator is a deep neural network that takes random noise as input to generate realistic data samples like images or text.
- It learns the underlying data patterns by adjusting its internal paramters during training through backpropagation.
- Its objective is to produce samples that the discriminator classifier as real.
  - J(G) = -1/mΣlogD(G(z(i)))
  - where,
    - J(G) - measures how well the generator is fooling the discriminator.
    - G(z(i)) - is the generated sample from random noise z(i).
    - D(G(z(i))) - is the discrimintor's estimated probability that the generated sample is real.
#### 2. Discriminator Model:
- The discriminator acts as a binary classifier helps in distinguishing between real and generated data.
- It learns to improve its classification ability through training, refining its parameters to detect fake samples more accurately.
- When dealing with image data, the discriminator uses convolutional layers or other relevant architectures which help to extract features and enhance the model's ability.
  - J(D) = -1/mΣlogD(x(i)) - 1/mΣlog(1-D(G(z(i))).
  - where,
    - J(D) - measures how well the discriminator classifier real and fake samples.
    - x(i) - is real data sample.
    - G(z(i)) - is a fake sample from the generator.
    - D(x(i)) - is the discriminator's probability that x(i) is real.
    - D(G(z(i))) - is the discriminator's probability that the fake sample is real.
- The discriminator wants to correctly classify real data as real (maximize logD(x(i))) and fake data as fake(maximizelog(1-D(G(z(i))).

### Types of GAN:
- We can have different types of GAN models based on the way the generator and the discriminator networks interact with each other.
#### 1. Vanilla GAN:
- The vanilla GAN is the simplest type of GAN made up of the generator and discriminator.
- Here the classification and generation of images is done by the generator and discriminator internally with the use of multilayer perceptrons.
- The generator captures the data distribution meanwhile, the discriminator tries to find the probability of the input belonging to a certain class, finally the feedback is sent to both the generator and discriminator after calculating the loss function.
- Hence the effort to minimize the loss comes into picture.
#### 2. Deep Convolutional GANs(DCGANs):
- DCGANs is the first GAN where the generator used deep convolutional network, hence generating high resolution and quality images to be diffrentiated.
- ReLU activation is used in Generator all layers except last one where Tanh activation is used.
- Leaky-ReLU activation function is used in Discriminator all layers. Adam optimizer is used with a learning rate of 0.0002.
#### 3. Conditional GANs:
- Conditional GAN includes additional condition information like class labels, attributes or even other data samples, into both generator and discriminator.
- With the help of these conditioning information, Conditional GANs provide us the control over the characteristic of the generated output.
#### 4. CycleGAN:
- This GAN is made for image-to-image translations, meaning one image to be mapped with another image.
- CycleGAns are designed for unpaired image-to-image translation tasks where there is no relation between the input and output images.
- A cycle consistency loss is used to ensure that translating from one domain to another and back again produces consistent results.
#### 5. Progressive GANs:
- ProGANs generate high-resolution images by progressively increasing the resolution of both the generator and discriminator during training.
- With the approach, you can create more detailed and higher-quality images.
#### 6. Laplician Pyramid GAN(LAPGAN):
- Laplacian Pyramid GAN is a type of GAN that uses a multi-resolution approach to generate high-quality images.
- It uses a Laplacian pyramid framework where images are generated by multiple scales.
- LAPGANs are mainly effective in creating and realistic images as compared to standard GANs.
#### 7. StyleGANs:
- StyleGAns is specially designed for generating photo-realistic high-quality images.
- They introduced some innovative techniques for improved image synthesis and have some better control over specific attributes.

### Implementation of GAN:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# Defining the Image Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])


# Loading the CIFAR-10 Dataset
train_dataset = datasets.CIFAR10(root='./data',\
              train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(train_dataset, \
                                batch_size=32, shuffle=True)

# Defining GAN Hyperparamters:
latent_dim = 100
lr = 0.0002
beta1 = 0.5
beta2 = 0.999
num_epochs = 10


# Building the Generator
class Generator(nn.Module):
    def __init__(self, latent_dim):
        super(Generator, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128 * 8 * 8),
            nn.ReLU(),
            nn.Unflatten(1, (128, 8, 8)),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128, momentum=0.78),
            nn.ReLU(),
            nn.Upsample(scale_factor=2),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64, momentum=0.78),
            nn.ReLU(),
            nn.Conv2d(64, 3, kernel_size=3, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        img = self.model(z)
        return img


# Building the Discriminator
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()

        self.model = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, stride=2, padding=1),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.25),
        nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
        nn.ZeroPad2d((0, 1, 0, 1)),
        nn.BatchNorm2d(64, momentum=0.82),
        nn.LeakyReLU(0.25),
        nn.Dropout(0.25),
        nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
        nn.BatchNorm2d(128, momentum=0.82),
        nn.LeakyReLU(0.2),
        nn.Dropout(0.25),
        nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
        nn.BatchNorm2d(256, momentum=0.8),
        nn.LeakyReLU(0.25),
        nn.Dropout(0.25),
        nn.Flatten(),
        nn.Linear(256 * 5 * 5, 1),
        nn.Sigmoid()
    )

    def forward(self, img):
        validity = self.model(img)
        return validity

# Initializing GAN Components
generator = Generator(latent_dim).to(device)
discriminator = Discriminator().to(device)

adversarial_loss = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters()\
                         , lr=lr, betas=(beta1, beta2))
optimizer_D = optim.Adam(discriminator.parameters()\
                         , lr=lr, betas=(beta1, beta2))


# Training the GAN
for epoch in range(num_epochs):
    for i, batch in enumerate(dataloader):
       
        real_images = batch[0].to(device) 
       
        valid = torch.ones(real_images.size(0), 1, device=device)
        fake = torch.zeros(real_images.size(0), 1, device=device)
       
        real_images = real_images.to(device)

        optimizer_D.zero_grad()
       
        z = torch.randn(real_images.size(0), latent_dim, device=device)
      
        fake_images = generator(z)

        real_loss = adversarial_loss(discriminator\
                                     (real_images), valid)
        fake_loss = adversarial_loss(discriminator\
                                     (fake_images.detach()), fake)
        d_loss = (real_loss + fake_loss) / 2
    
        d_loss.backward()
        optimizer_D.step()

        optimizer_G.zero_grad()
      
        gen_images = generator(z)
        
        g_loss = adversarial_loss(discriminator(gen_images), valid)
        g_loss.backward()
        optimizer_G.step()
       
        if (i + 1) % 100 == 0:
            print(
                f"Epoch [{epoch+1}/{num_epochs}]\
                        Batch {i+1}/{len(dataloader)} "
                f"Discriminator Loss: {d_loss.item():.4f} "
                f"Generator Loss: {g_loss.item():.4f}"
            )
    if (epoch + 1) % 10 == 0:
        with torch.no_grad():
            z = torch.randn(16, latent_dim, device=device)
            generated = generator(z).detach().cpu()
            grid = torchvision.utils.make_grid(generated,\
                                        nrow=4, normalize=True)
            plt.imshow(np.transpose(grid, (1, 2, 0)))
            plt.axis("off")
            plt.show()

100%|███████████████████████████████████████████████████████████████████████████████| 170M/170M [00:22<00:00, 7.71MB/s]
C:\Users\techs\anaconda3\New folder\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Epoch [1/10]                        Batch 100/1563 Discriminator Loss: 0.6059 Generator Loss: 1.2442
Epoch [1/10]                        Batch 200/1563 Discriminator Loss: 0.6900 Generator Loss: 0.8537
Epoch [1/10]                        Batch 300/1563 Discriminator Loss: 0.5387 Generator Loss: 1.5283
Epoch [1/10]                        Batch 400/1563 Discriminator Loss: 0.4077 Generator Loss: 1.5629
Epoch [1/10]                        Batch 500/1563 Discriminator Loss: 0.5304 Generator Loss: 2.0331
